### [Heteroscedasticity in Financial Econometrics](https://medium.com/financial-engineering/944ad2081573)

> *Financial Econometrics: Part 05*

- Ordinary Least Squares (OLS)
- Gauss-Markov Theorem
- OLS estimator is **BLUE**: the *B*est *L*inear *U*nbiased *E*stimator

**Homoscedasticity ("Same Scatter"):** This is the ideal state. It means that the variability of the error term (the “scatter” of the data points around the regression line) is the same, regardless of the value of the independent variable.

**Heteroscedasticity ("Different Scatter"):** This is the problematic, real-world scenario. It means the variance of the error term changes as the independent variable changes.

In [1]:
import sys

import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import numpy as np
import pandas as pd

import statsmodels.api as sm
import statsmodels.stats.api as sms

URL = 'https://simplified-zone.com/wp-content/uploads/2025/11/M2_data.csv'

IN_COLAB = 'google.colab' in sys.modules

np.random.seed(99)
np.set_printoptions(suppress=True, precision=4)

pd.set_option('display.width', 1000)
pd.set_option('display.max_rows', None)
pd.set_option('display.precision', 4)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.4f}'.format)

%autosave 15

Autosaving every 15 seconds


#### **1. LOAD AND PREPARE THE DATA**

In [2]:
print("--- 1. Loading Data ---")
# Load the dataset
data = pd.read_csv(URL)

# Convert 'Date' to datetime objects
data['Date'] = pd.to_datetime(data['Date'])

# Set the Date as the index
data = data.set_index('Date')

display(data.head(10))

# Define our dependent (Y) and independent (X) variables
# Y = DXY (US Dollar Index)
# X = Everything else (except YEAR)
dependent_var = 'DXY'
independent_vars = ['METALS', 'OIL', 'US_STK', 'INTL_STK', 'X10Y_TBY', 'EURUSD']

# Drop any rows with missing data
data_clean = data[[dependent_var] + independent_vars].dropna()

# Prepare Y and X for statsmodels
Y = data_clean[dependent_var]
X = data_clean[independent_vars]

# Add a constant (intercept) to the X matrix
X = sm.add_constant(X)

print(f"Data loaded. Modeling {dependent_var} with {len(independent_vars)} predictors.")
print(f"Total observations: {len(Y)}")
print("-" * 30 + "\n")

--- 1. Loading Data ---


,DXY,METALS,OIL,US_STK,INTL_STK,X13W_TB,X10Y_TBY,EURUSD,YEAR
Date,,,,,,,,,
2016-01-04,0.0024,0.0243,-0.0076,-0.0140,-0.0198,0.0473,-0.0106,-0.0073,2016
2016-01-05,0.0054,-0.0047,-0.0215,0.0017,-0.0013,0.3226,0.0013,-0.0024,2016
2016-01-06,-0.0022,0.0136,-0.0556,-0.0126,-0.0152,0.0000,-0.0316,-0.0070,2016
2016-01-07,-0.0097,0.0352,-0.0206,-0.0240,-0.0193,-0.0732,-0.0110,0.0025,2016
2016-01-08,0.0033,-0.0281,-0.0033,-0.0110,-0.0105,0.0000,-0.0107,0.0136,2016
2016-01-11,0.0019,-0.0577,-0.0528,0.0010,-0.0035,-0.0789,0.0131,0.0014,2016
2016-01-12,0.0024,-0.0345,-0.0309,0.0081,0.0040,0.2000,-0.0259,-0.0076,2016
2016-01-13,-0.0004,0.0047,0.0013,-0.0249,-0.0101,0.0238,-0.0171,-0.0013,2016
2016-01-14,0.0016,-0.0355,0.0236,0.0164,0.0058,0.1163,0.0155,0.0039,2016


Data loaded. Modeling DXY with 6 predictors.
Total observations: 250
------------------------------



#### **2. RUN INITIAL OLS REGRESSION (MODEL 1)**

In [3]:
print("--- 2. Running Initial OLS Regression (Model 1) ---")
ols_model = sm.OLS(Y, X)
ols_results = ols_model.fit()

# Print the OLS summary
print(ols_results.summary())
print("-" * 30 + "\n")

--- 2. Running Initial OLS Regression (Model 1) ---
                            OLS Regression Results                            
Dep. Variable:                    DXY   R-squared:                       0.381
Model:                            OLS   Adj. R-squared:                  0.366
Method:                 Least Squares   F-statistic:                     24.96
Date:                Wed, 22 Apr 2026   Prob (F-statistic):           5.21e-23
Time:                        13:37:25   Log-Likelihood:                 1043.0
No. Observations:                 250   AIC:                            -2072.
Df Residuals:                     243   BIC:                            -2047.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------


#### **3. TEST FOR HETEROSCEDASTICITY (BREUSCH-PAGAN TEST)**

In [4]:
print("--- 3. Testing for Heteroscedasticity (Breusch-Pagan) ---")

# Get the residuals from our OLS model
ols_residuals = ols_results.resid

# Run the Breusch-Pagan test
# The 'het_breuschpagan' function takes the residuals and the X matrix
# It returns:
# 1. LM statistic
# 2. p-value for the LM statistic
# 3. F-statistic
# 4. p-value for the F-statistic
bp_test = sms.het_breuschpagan(ols_residuals, X)
labels = ['LM Statistic', 'LM Test p-value', 'F-Statistic', 'F-Test p-value']

# Print the results as a clean dictionary
bp_results = dict(zip(labels, bp_test))
print("Breusch-Pagan Test Results:")
print(bp_results)

# Interpret the results
if bp_results['LM Test p-value'] < 0.05:
  print("\nInterpretation: The p-value is less than 0.05.")
  print("We REJECT the null hypothesis of homoscedasticity.")
  print("Conclusion: HETEROSCEDASTICITY IS PRESENT.")
  print("The OLS standard errors are unreliable. We must use WLS.")
else:
  print("\nInterpretation: The p-value is greater than 0.05.")
  print("We FAIL to reject the null hypothesis of homoscedasticity.")
  print("Conclusion: No significant heteroscedasticity detected.")

print("-" * 30 + "\n")

--- 3. Testing for Heteroscedasticity (Breusch-Pagan) ---
Breusch-Pagan Test Results:
{'LM Statistic': np.float64(12.906784427399925), 'LM Test p-value': np.float64(0.04454033207681837), 'F-Statistic': np.float64(2.2047225942221633), 'F-Test p-value': np.float64(0.04325736507354383)}

Interpretation: The p-value is less than 0.05.
We REJECT the null hypothesis of homoscedasticity.
Conclusion: HETEROSCEDASTICITY IS PRESENT.
The OLS standard errors are unreliable. We must use WLS.
------------------------------



#### **4. PERFORM WEIGHTED LEAST SQUARES (WLS) (MODEL 2)**

In [5]:
# We only do this if heteroscedasticity was found, but we'll run it
# for demonstration purposes regardless.

print("--- 4. Running Weighted Least Squares (WLS) Regression ---")

# Step 4a: Get squared residuals from OLS
ols_resid_sq = ols_residuals**2

# Step 4b: Run auxiliary regression to model the variance
# We model the log of the squared residuals to ensure positive variance estimates
# Note: We add a small constant (1e-8) to avoid log(0)
aux_Y = np.log(ols_resid_sq + 1e-8)
aux_X = X # Use the same X matrix

aux_model = sm.OLS(aux_Y, aux_X)
aux_results = aux_model.fit()

# Step 4c: Get the fitted values from the auxiliary regression
# These are our estimates for log(variance)
log_variance_hat = aux_results.fittedvalues

# Step 4d: Calculate the weights
# The variance estimate is exp(log_variance_hat)
# The weight is 1 / variance
weights = 1.0 / np.exp(log_variance_hat)

# Step 4e: Run the final WLS regression using these weights
wls_model = sm.WLS(Y, X, weights=weights)
wls_results = wls_model.fit()

print("WLS Model (Model 2) complete. See summary below.")
print("-" * 30 + "\n")

--- 4. Running Weighted Least Squares (WLS) Regression ---
WLS Model (Model 2) complete. See summary below.
------------------------------



#### **5. COMPARE OLS AND WLS RESULTS**

In [6]:
print("--- 5. Final Model Comparison ---")

print("\n*** OLS MODEL (MODEL 1) SUMMARY ***")
print("(Unreliable Standard Errors)")
display(ols_results.summary())

print("\n\n*** WLS MODEL (MODEL 2) SUMMARY ***")
print("(Reliable Standard Errors)")
display(wls_results.summary())

print("\n\n--- End of Analysis ---")

--- 5. Final Model Comparison ---

*** OLS MODEL (MODEL 1) SUMMARY ***
(Unreliable Standard Errors)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    DXY   R-squared:                       0.381
Model:                            OLS   Adj. R-squared:                  0.366
Method:                 Least Squares   F-statistic:                     24.96
Date:                Wed, 22 Apr 2026   Prob (F-statistic):           5.21e-23
Time:                        13:38:04   Log-Likelihood:                 1043.0
No. Observations:                 250   AIC:                            -2072.
Df Residuals:                     243   BIC:                            -2047.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0002      0.000      0.740      0.460      -0.000       0.001
METALS        -0.0538      0.009     -6.164      0.000      -0.071      -0.037
OIL            0.0194      0.009      2.234      0.026       0.002       0.036
US_STK         0.3060      0.057      5.344      0.000       0.193       0.419
INTL_STK      -0.3337      0.045     -7.426      0.000      -0.422      -0.245
X10Y_TBY       0.0196      0.011      1.735      0.084      -0.003       0.042
EURUSD        -0.0691      0.045     -1.533      0.127      -0.158       0.020
==============================================================================
Omnibus:                        9.919   Durbin-Watson:                   2.184
Prob(Omnibus):                  0.007   Jarque-Bera (JB):               18.481
Skew:                           0.131   Prob(JB):                     9.70e-05
Kurtosis:                       4.306   Cond. No.                         292.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""



*** WLS MODEL (MODEL 2) SUMMARY ***
(Reliable Standard Errors)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:                    DXY   R-squared:                       0.379
Model:                            WLS   Adj. R-squared:                  0.364
Method:                 Least Squares   F-statistic:                     24.72
Date:                Wed, 22 Apr 2026   Prob (F-statistic):           8.08e-23
Time:                        13:38:04   Log-Likelihood:                 1049.8
No. Observations:                 250   AIC:                            -2086.
Df Residuals:                     243   BIC:                            -2061.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0002      0.000      0.668      0.505      -0.000       0.001
METALS        -0.0532      0.008     -6.487      0.000      -0.069      -0.037
OIL            0.0181      0.008      2.145      0.033       0.001       0.035
US_STK         0.2736      0.055      4.954      0.000       0.165       0.382
INTL_STK      -0.2846      0.046     -6.239      0.000      -0.374      -0.195
X10Y_TBY       0.0261      0.011      2.468      0.014       0.005       0.047
EURUSD        -0.0829      0.044     -1.900      0.059      -0.169       0.003
==============================================================================
Omnibus:                        4.527   Durbin-Watson:                   2.290
Prob(Omnibus):                  0.104   Jarque-Bera (JB):                5.589
Skew:                           0.102   Prob(JB):                       0.0612
Kurtosis:                       3.704   Cond. No.                         300.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""



--- End of Analysis ---
